# Day 7 · 10 minute map — 10 นาทีจากจุดดูดาว 83 แห่ง ไปไกลแค่ไหน

**#30DayMapChallenge 2026 · วันที่ 7 พ.ย. 2569** · ธีม *"make your map in 10 minutes"*

กติกา: จับเวลาจริงตั้งแต่บรรทัดแรกถึงไฟล์รูปที่ยอมรับได้ · **ผล: 7 นาที 29 วินาที** (23:45:22 → 23:52:51)
คอมคำนวณเสร็จใน ~1.1 วินาที ที่เหลือคือคิดหัวข้อกับแก้บั๊ก 4 ตัว

> โน้ตบุ๊กนี้คือสคริปต์เดียวที่รันจริงในวันนั้น รวมถึงรอบที่แก้บั๊กแล้ว (เวอร์ชันสุดท้าย)
> ถ้าอยากลองจับเวลาแบบผม ให้ restart kernel แล้วรันทั้งหมดรวดเดียว


In [ ]:
# -*- coding: utf-8 -*-
"""Day 7 · 10 minute map — เขียนและรันครั้งเดียว ไม่มีรอบแก้ความสวย
โจทย์สองชั้น: ทำแผนที่ภายใน 10 นาที และแผนที่นั้นเล่าเรื่อง "10 นาทีไปไกลแค่ไหน"
"""
import os, time, json
import numpy as np, pandas as pd
import geopandas as gpd
from shapely.geometry import box
from shapely.ops import unary_union
from pyproj import CRS
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.collections import PatchCollection
from matplotlib.patches import Circle

T0 = time.time()
WORK = r"D:\GIS30Days"; REPO = os.path.join(WORK, "geo-notes")
CACHE = os.path.join(WORK, "cache", "day06"); OUT = os.path.join(WORK, "out", "day07")
os.makedirs(OUT, exist_ok=True)
for f in sorted(os.listdir(CACHE)):
    if f.endswith(".ttf"): font_manager.fontManager.addfont(os.path.join(CACHE, f))
plt.rcParams["font.family"] = "Trirong"
log = lambda m: print(f"[{time.time()-T0:5.1f}s] {m}", flush=True)

# 1) ข้อมูล: 83 จุดดูดาว NARIT + ขอบประเทศ
df = pd.read_csv(os.path.join(REPO, "data", "darksky_thailand.csv"))
countries = gpd.read_file(os.path.join(CACHE, "ne_10m_admin_0_countries.shp"))
th = countries[countries.ISO_A3 == "THA"][["geometry"]].to_crs(
    CRS.from_proj4("+proj=eqdc +lat_0=14 +lon_0=101 +lat_1=8 +lat_2=20 +units=m"))
pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326").to_crs(th.crs)
log(f"โหลดข้อมูล {len(df)} จุด + ขอบประเทศ เรียบร้อย")

# 2) "10 นาที" ของสามพาหนะ (ระยะ = ความเร็วเฉลี่ย x 1/6 ชม.)
MODES = [("เดิน 10 นาที", 4.8, "#7c3aed", 0.10),
         ("จักรยาน 10 นาที", 15.0, "#0891b2", 0.12),
         ("รถยนต์ในเมือง 10 นาที", 54.0, "#f59e0b", 0.14)]
R = {n: v / 3.6 * 600.0 for n, v, c, a in MODES}      # เมตร ใน 10 นาที
cov = {}
for n, v, c, a in MODES:
    u = unary_union(pts.geometry.buffer(R[n]))
    cov[n] = u.area / 1e6
log(f"คำนวณพื้นที่คลุม 3 ระดับ: " + ", ".join(f"{k}={vv:.1f}k ตร.กม." for k, vv in cov.items()))

# 3) วาดครั้งเดียว
fig, ax = plt.subplots(figsize=(9.5, 11), facecolor="#0f172a"); ax.set_facecolor("#0f172a")
ax.set_aspect("equal"); ax.axis("off")
th.boundary.plot(ax=ax, color="#475569", lw=0.8)
for n, v, c, a in reversed(MODES):
    for geom in pts.geometry:
        ax.add_patch(Circle((geom.x, geom.y), R[n], facecolor=c, alpha=a, edgecolor=c, lw=0.35))
ax.scatter(pts.geometry.x, pts.geometry.y, s=6, color="#f8fafc", zorder=6)
x0, y0, x1, y1 = th.total_bounds; pad = 60_000
ax.set_xlim(x0-pad, x1+pad); ax.set_ylim(y0-pad, y1+pad)
ax.set_title("Day 7 · 10 minute map — จากจุดดูดาว 83 แห่ง 10 นาทีไปไกลแค่ไหน",
             color="white", fontsize=13, pad=14, fontweight="bold")
leg = "\n".join(f"■ {n}: รัศมี {R[n]/1000:.1f} กม. → คลุมรวม {cov[n]:,.0f} ตร.กม." for n, v, c, a in MODES)
ax.text(0.02, 0.03, leg, transform=ax.transAxes, color="#cbd5e1", fontsize=9.5, va="bottom",
        linespacing=1.7)
ax.text(0.60, 0.03, f"ใช้เวลาสร้างแผนที่ {time.time()-T0:.0f} วินาที (นับรวมรอบแก้บั๊ก)", transform=ax.transAxes,
        color="#94a3b8", fontsize=8.5, ha="right", va="bottom")
# panel ขยาย: บริเวณกรุงเทพ–ฉะเชิงเทรา–เขาใหญ่ ที่วง 10 นาทีอ่านออก
axz = fig.add_axes([0.635, 0.075, 0.335, 0.235]); axz.set_facecolor("#0b1222")
axz.set_aspect("equal")
for s_ in axz.spines.values(): s_.set_color("#334155")
axz.set_xticks([]); axz.set_yticks([])
BX0, BX1, BY0, BY1 = 99.3, 102.4, 13.2, 14.9
sub = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326")
sub = sub[sub.longitude.between(BX0, BX1) & sub.latitude.between(BY0, BY1)].to_crs(th.crs)
th.clip(box := __import__("shapely.geometry", fromlist=["box"]).box(BX0, BY0, BX1, BY1)).to_crs(th.crs).boundary.plot(ax=axz, color="#475569", lw=0.6)
for n, v, c, a in reversed(MODES):
    for geom in sub.geometry:
        axz.add_patch(Circle((geom.x, geom.y), R[n], facecolor=c, alpha=a+0.06, edgecolor=c, lw=0.5))
axz.scatter(sub.geometry.x, sub.geometry.y, s=14, color="#f8fafc", zorder=6)
zx = np.array([g.x for g in sub.geometry]); zy = np.array([g.y for g in sub.geometry])
axz.set_xlim(zx.min()-14_000, zx.max()+14_000); axz.set_ylim(zy.min()-14_000, zy.max()+14_000)
axz.set_facecolor("#0b1222")
axz.set_title(f"ขยาย {len(sub)} จุด: กรุงเทพ–ฉะเชิงเทรา–เขาใหญ่", color="#e2e8f0", fontsize=9.5)
p = os.path.join(OUT, "day07_10min.png")
fig.savefig(p, dpi=150, facecolor="#0f172a", bbox_inches="tight")
log(f"savefig → {p} ({os.path.getsize(p)/1e6:.2f} MB)")
json.dump({"seconds": round(time.time()-T0, 1), "coverage_thousand_km2": {k: round(v, 1) for k, v in cov.items()},
           "radii_m": {k: round(R[k]) for k in R}}, open(os.path.join(OUT, "day07_stats.json"), "w"),
          ensure_ascii=False, indent=1)
log(f"รวมเวลา {time.time()-T0:.1f} วินาที = {(time.time()-T0)/60:.2f} นาที")
